# AutoMixAI – Multi-Task Audio Model Training

This notebook trains multiple models for the AutoMixAI project:

| Task | Dataset | Author/Source |
|------|---------|---------------|
| **Beat Detection Enhancement** | Freesound Audio Tagging 2019 | `yuxuanxue/freesound-audio-tagging-2019` |
| **Instrument Classification** | NSynth Music Dataset | `anubhavchhabra/nsynth-music-dataset` |
| **Drum Classification** | Drum Kit Sound Samples | `sparshgupta/drum-kit-sound-samples` |
| **MIDI Generation Features** | Lakh MIDI Dataset Clean | `federicodellellis/lakh-midi-dataset-clean` |
| **Mixing Intelligence (Tags)** | MagnaTagATune | `shrirangmahajan/magnatagatune` |
| **Mixing Intelligence (Meta)** | Spotify Million Song Dataset | `notshrirang/spotify-million-song-dataset` |

## Kaggle Setup

Add these datasets to your Kaggle notebook:
```
+ Add Data → Search each dataset by author/name above
```

**Note**: Some datasets are large. You may need Kaggle GPU/TPU runtime.

---

## Outputs

| File | Description |
|------|-------------|
| `nsynth_classifier.h5` | NSynth-based instrument classifier |
| `nsynth_scaler.pkl` | Scaler for instrument features |
| `drum_classifier.h5` | Drum type classifier (kick/snare/hihat/etc) |
| `drum_scaler.pkl` | Scaler for drum features |
| `onset_detector.h5` | Enhanced onset/beat detector |
| `tag_predictor.h5` | Multi-label tag predictor for mixing |
| `tag_scaler.pkl` | Scaler for tag features |
| `midi_patterns.pkl` | Extracted MIDI rhythm patterns |
| `song_metadata.pkl` | Spotify song metadata for mixing |

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# CELL 1: Install Dependencies
# ══════════════════════════════════════════════════════════════════════

import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'tensorflow', 'librosa', 'scikit-learn', 'pandas',
                'numpy', 'matplotlib', 'seaborn', 'pretty_midi', 'tqdm'], check=True)
print('All packages ready.')

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# CELL 2: Imports
# ══════════════════════════════════════════════════════════════════════

import os
import json
import glob
import joblib
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

import librosa
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, callbacks

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder, MultiLabelBinarizer
from sklearn.metrics import classification_report, confusion_matrix, f1_score

print(f'TensorFlow: {tf.__version__}')
print(f'Librosa: {librosa.__version__}')
print(f'GPU: {tf.config.list_physical_devices("GPU")}')

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# CELL 3: Configuration & Dataset Paths
# ══════════════════════════════════════════════════════════════════════

# Audio processing constants (match AutoMixAI backend)
SAMPLE_RATE = 22050
HOP_LENGTH = 512
N_MELS = 128
N_MFCC = 20
N_FFT = 2048

# Kaggle dataset paths
# ─────────────────────────────────────────────────────────────────────
# IMPORTANT: Add these datasets to your Kaggle notebook!
#
# Dataset                        │ Kaggle Author/Path
# ───────────────────────────────┼─────────────────────────────────────
# Freesound Audio Tagging 2019   │ yuxuanxue/freesound-audio-tagging-2019
# NSynth Music Dataset           │ anubhavchhabra/nsynth-music-dataset
# Drum Kit Sound Samples         │ sparshgupta/drum-kit-sound-samples
# Lakh MIDI Dataset Clean        │ federicodellellis/lakh-midi-dataset-clean
# MagnaTagATune                  │ shrirangmahajan/magnatagatune
# Spotify Million Song Dataset   │ notshrirang/spotify-million-song-dataset
# ─────────────────────────────────────────────────────────────────────

KAGGLE_INPUT = '/kaggle/input'

PATHS = {
    'freesound': os.path.join(KAGGLE_INPUT, 'freesound-audio-tagging-2019'),
    'nsynth': os.path.join(KAGGLE_INPUT, 'nsynth-music-dataset'),
    'drums': os.path.join(KAGGLE_INPUT, 'drum-kit-sound-samples'),
    'lakh_midi': os.path.join(KAGGLE_INPUT, 'lakh-midi-dataset-clean'),
    'magnatagatune': os.path.join(KAGGLE_INPUT, 'magnatagatune'),
    'spotify_msd': os.path.join(KAGGLE_INPUT, 'spotify-million-song-dataset'),
}

# Check which datasets are available
print('Dataset availability:')
for name, path in PATHS.items():
    exists = os.path.exists(path)
    status = '✓' if exists else '✗ (add dataset)'
    print(f'  {name:20} {status}')

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# CELL 4: Feature Extraction Utilities
# ══════════════════════════════════════════════════════════════════════

def extract_audio_features(y, sr=SAMPLE_RATE):
    """Extract comprehensive audio features for classification tasks."""
    features = []
    
    # Spectral features
    spec_cent = librosa.feature.spectral_centroid(y=y, sr=sr, hop_length=HOP_LENGTH)
    spec_bw = librosa.feature.spectral_bandwidth(y=y, sr=sr, hop_length=HOP_LENGTH)
    spec_rolloff = librosa.feature.spectral_rolloff(y=y, sr=sr, hop_length=HOP_LENGTH)
    spec_flat = librosa.feature.spectral_flatness(y=y, hop_length=HOP_LENGTH)
    
    features.extend([spec_cent.mean(), spec_cent.std()])
    features.extend([spec_bw.mean(), spec_bw.std()])
    features.extend([spec_rolloff.mean(), spec_rolloff.std()])
    features.extend([spec_flat.mean(), spec_flat.std()])
    
    # Chroma
    chroma = librosa.feature.chroma_stft(y=y, sr=sr, hop_length=HOP_LENGTH)
    features.extend([chroma.mean(), chroma.std()])
    
    # ZCR
    zcr = librosa.feature.zero_crossing_rate(y, hop_length=HOP_LENGTH)
    features.extend([zcr.mean(), zcr.std()])
    
    # RMS
    rms = librosa.feature.rms(y=y, hop_length=HOP_LENGTH)
    features.extend([rms.mean(), rms.std()])
    
    # MFCCs
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=N_MFCC, hop_length=HOP_LENGTH)
    for i in range(N_MFCC):
        features.extend([mfcc[i].mean(), mfcc[i].std()])
    
    # Tempo
    tempo, _ = librosa.beat.beat_track(y=y, sr=sr, hop_length=HOP_LENGTH)
    features.append(float(tempo))
    
    return np.array(features, dtype=np.float32)


def extract_mel_spectrogram(y, sr=SAMPLE_RATE, n_mels=N_MELS, fixed_length=128):
    """Extract mel spectrogram with fixed time dimension."""
    mel = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=n_mels, hop_length=HOP_LENGTH)
    mel_db = librosa.power_to_db(mel, ref=np.max)
    
    # Pad or truncate to fixed length
    if mel_db.shape[1] < fixed_length:
        mel_db = np.pad(mel_db, ((0, 0), (0, fixed_length - mel_db.shape[1])), mode='constant')
    else:
        mel_db = mel_db[:, :fixed_length]
    
    return mel_db.astype(np.float32)


def load_audio_safe(path, sr=SAMPLE_RATE, duration=None):
    """Safely load audio file with error handling."""
    try:
        y, _ = librosa.load(path, sr=sr, duration=duration, mono=True)
        if len(y) == 0:
            return None
        return y
    except Exception as e:
        return None


print('Feature extraction utilities loaded.')

---

# 1. Drum Classification Model

**Dataset**: `shashank4a4/drum-sound-classification-dataset`

Classifies drum sounds into categories: kick, snare, hihat, tom, cymbal, etc.

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# CELL 5: Load Drum Dataset
# ══════════════════════════════════════════════════════════════════════

DRUM_PATH = PATHS['drums']

if os.path.exists(DRUM_PATH):
    # Discover structure
    print('Drum Kit Sound Samples dataset structure:')
    for root, dirs, files in os.walk(DRUM_PATH):
        level = root.replace(DRUM_PATH, '').count(os.sep)
        indent = ' ' * 2 * level
        folder = os.path.basename(root)
        audio_count = len([f for f in files if f.endswith(('.wav', '.mp3', '.ogg'))])
        if audio_count > 0 or level < 2:
            print(f'{indent}{folder}/ ({audio_count} audio files)' if audio_count else f'{indent}{folder}/')
        if level > 2:
            break
else:
    print(f'⚠ Dataset not found at {DRUM_PATH}')
    print('  Add: sparshgupta/drum-kit-sound-samples')

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# CELL 6: Prepare Drum Features
# ══════════════════════════════════════════════════════════════════════

def prepare_drum_dataset(base_path, max_per_class=500):
    """Load drum samples and extract features."""
    X, y = [], []
    
    # Look for class folders
    class_folders = [d for d in os.listdir(base_path) 
                     if os.path.isdir(os.path.join(base_path, d))]
    
    if not class_folders:
        # Try one level deeper
        for subdir in os.listdir(base_path):
            subpath = os.path.join(base_path, subdir)
            if os.path.isdir(subpath):
                class_folders = [os.path.join(subdir, d) for d in os.listdir(subpath)
                                if os.path.isdir(os.path.join(subpath, d))]
                if class_folders:
                    break
    
    print(f'Found {len(class_folders)} drum classes')
    
    for class_folder in tqdm(class_folders, desc='Processing drum classes'):
        class_path = os.path.join(base_path, class_folder)
        class_name = os.path.basename(class_folder).lower()
        
        # Normalize class names
        if 'kick' in class_name or 'bass' in class_name:
            label = 'kick'
        elif 'snare' in class_name:
            label = 'snare'
        elif 'hihat' in class_name or 'hi-hat' in class_name or 'hh' in class_name:
            label = 'hihat'
        elif 'tom' in class_name:
            label = 'tom'
        elif 'cymbal' in class_name or 'crash' in class_name or 'ride' in class_name:
            label = 'cymbal'
        elif 'clap' in class_name:
            label = 'clap'
        elif 'perc' in class_name:
            label = 'percussion'
        else:
            label = class_name
        
        audio_files = glob.glob(os.path.join(class_path, '**', '*.wav'), recursive=True)
        audio_files += glob.glob(os.path.join(class_path, '**', '*.mp3'), recursive=True)
        audio_files = audio_files[:max_per_class]
        
        for audio_file in audio_files:
            audio = load_audio_safe(audio_file, duration=1.0)  # Drum hits are short
            if audio is not None and len(audio) > SAMPLE_RATE // 10:  # At least 0.1s
                try:
                    features = extract_audio_features(audio)
                    if not np.any(np.isnan(features)) and not np.any(np.isinf(features)):
                        X.append(features)
                        y.append(label)
                except:
                    pass
    
    return np.array(X), np.array(y)


if os.path.exists(DRUM_PATH):
    X_drums, y_drums = prepare_drum_dataset(DRUM_PATH)
    print(f'\nDrum dataset: {X_drums.shape[0]} samples, {X_drums.shape[1]} features')
    print(f'Classes: {np.unique(y_drums)}')
    print(f'Distribution: {pd.Series(y_drums).value_counts().to_dict()}')
else:
    print('Skipping drum dataset (not available)')
    X_drums, y_drums = None, None

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# CELL 7: Train Drum Classifier
# ══════════════════════════════════════════════════════════════════════

if X_drums is not None and len(X_drums) > 100:
    # Encode labels
    drum_le = LabelEncoder()
    y_drums_enc = drum_le.fit_transform(y_drums)
    n_drum_classes = len(drum_le.classes_)
    print(f'Drum classes ({n_drum_classes}): {list(drum_le.classes_)}')
    
    # Split
    X_train, X_test, y_train, y_test = train_test_split(
        X_drums, y_drums_enc, test_size=0.2, random_state=42, stratify=y_drums_enc)
    
    # Scale
    drum_scaler = StandardScaler()
    X_train_s = drum_scaler.fit_transform(X_train)
    X_test_s = drum_scaler.transform(X_test)
    
    # Build model
    drum_model = keras.Sequential([
        keras.Input(shape=(X_train_s.shape[1],)),
        layers.Dense(128, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.3),
        layers.Dense(64, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.2),
        layers.Dense(32, activation='relu'),
        layers.Dense(n_drum_classes, activation='softmax')
    ], name='drum_classifier')
    
    drum_model.compile(
        optimizer='adam',
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    
    # Train
    drum_history = drum_model.fit(
        X_train_s, y_train,
        validation_split=0.15,
        epochs=50,
        batch_size=32,
        callbacks=[
            callbacks.EarlyStopping(patience=10, restore_best_weights=True),
            callbacks.ReduceLROnPlateau(patience=5, factor=0.5)
        ],
        verbose=1
    )
    
    # Evaluate
    test_loss, test_acc = drum_model.evaluate(X_test_s, y_test, verbose=0)
    print(f'\nDrum classifier test accuracy: {test_acc:.4f}')
    
    # Save
    drum_model.save('drum_classifier.h5')
    joblib.dump(drum_scaler, 'drum_scaler.pkl')
    joblib.dump(drum_le, 'drum_labels.pkl')
    print('Saved: drum_classifier.h5, drum_scaler.pkl, drum_labels.pkl')
else:
    print('Skipping drum model training (insufficient data)')

---

# 2. NSynth Instrument Classification Model

**Dataset**: `anubhavchhabra/nsynth-music-dataset`

NSynth provides high-quality labeled instrumental notes. We train a model to classify instrument families.

*Note: Music Instruments dataset not available on Kaggle - using NSynth only.*

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# CELL 8: Music Instruments Dataset - SKIPPED
# ══════════════════════════════════════════════════════════════════════

# Music Instruments dataset not available on Kaggle
# Using NSynth dataset for instrument classification instead
print('Music Instruments dataset not available - using NSynth (Section 3) instead')

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# CELL 9: Prepare Instrument Features - SKIPPED
# ══════════════════════════════════════════════════════════════════════

# Skipped - see NSynth section below
pass

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# CELL 10: Train Instrument Classifier - SKIPPED
# ══════════════════════════════════════════════════════════════════════

# Skipped - see NSynth section below
pass

---

# 3. NSynth Instrument Family Classifier

**Dataset**: `anubhavchhabra/nsynth-music-dataset`

NSynth provides high-quality labeled instrumental notes. We train a model to classify instrument families and qualities.

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# CELL 11: Load NSynth Dataset (TFRecord format)
# ══════════════════════════════════════════════════════════════════════

NSYNTH_PATH = PATHS['nsynth']

# NSynth instrument families (11 classes)
NSYNTH_FAMILIES = ['bass', 'brass', 'flute', 'guitar', 'keyboard', 
                   'mallet', 'organ', 'reed', 'string', 'synth_lead', 'vocal']

def parse_nsynth_tfrecord(example_proto):
    """Parse a single NSynth TFRecord example."""
    features = {
        'audio': tf.io.FixedLenFeature([64000], tf.float32),  # 4 seconds at 16kHz
        'instrument_family': tf.io.FixedLenFeature([], tf.int64),
        'instrument_source': tf.io.FixedLenFeature([], tf.int64),
        'pitch': tf.io.FixedLenFeature([], tf.int64),
        'velocity': tf.io.FixedLenFeature([], tf.int64),
    }
    parsed = tf.io.parse_single_example(example_proto, features)
    return parsed['audio'], parsed['instrument_family']

if os.path.exists(NSYNTH_PATH):
    print('NSynth Music Dataset contents:')
    for item in os.listdir(NSYNTH_PATH):
        fpath = os.path.join(NSYNTH_PATH, item)
        size_mb = os.path.getsize(fpath) / 1024 / 1024 if os.path.isfile(fpath) else 0
        print(f'  {item} ({size_mb:.1f} MB)' if size_mb else f'  {item}/')
    
    # Find TFRecord files
    tfrecord_files = glob.glob(os.path.join(NSYNTH_PATH, '*.tfrecord'))
    print(f'\nFound {len(tfrecord_files)} TFRecord files')
    
    # Use validation set (smaller) or train set
    valid_tfrecord = [f for f in tfrecord_files if 'valid' in f.lower()]
    train_tfrecord = [f for f in tfrecord_files if 'train' in f.lower()]
    
    nsynth_tfrecord = valid_tfrecord[0] if valid_tfrecord else (train_tfrecord[0] if train_tfrecord else None)
    if nsynth_tfrecord:
        print(f'Using: {os.path.basename(nsynth_tfrecord)}')
else:
    print(f'⚠ Dataset not found at {NSYNTH_PATH}')
    print('  Add: anubhavchhabra/nsynth-music-dataset')
    nsynth_tfrecord = None

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# CELL 12: Prepare NSynth Features from TFRecord
# ══════════════════════════════════════════════════════════════════════

def extract_features_from_audio_array(audio_array, sr=16000):
    """Extract features from raw audio array (NSynth is 16kHz)."""
    # Resample to our standard sample rate if needed
    if sr != SAMPLE_RATE:
        audio_array = librosa.resample(audio_array, orig_sr=sr, target_sr=SAMPLE_RATE)
    
    return extract_audio_features(audio_array, sr=SAMPLE_RATE)


def prepare_nsynth_from_tfrecord(tfrecord_path, max_samples=3000):
    """Load NSynth samples from TFRecord and extract features."""
    X, y = [], []
    
    # Create dataset from TFRecord
    dataset = tf.data.TFRecordDataset(tfrecord_path)
    dataset = dataset.map(parse_nsynth_tfrecord)
    
    print(f'Processing up to {max_samples} samples from TFRecord...')
    
    count = 0
    for audio, instrument_family in tqdm(dataset, total=max_samples):
        if count >= max_samples:
            break
        
        try:
            # Convert tensor to numpy
            audio_np = audio.numpy().astype(np.float32)
            family_idx = int(instrument_family.numpy())
            
            # Extract features
            features = extract_features_from_audio_array(audio_np, sr=16000)
            
            if not np.any(np.isnan(features)) and not np.any(np.isinf(features)):
                X.append(features)
                y.append(family_idx)
                count += 1
        except Exception as e:
            pass
    
    return np.array(X), np.array(y)


if nsynth_tfrecord is not None:
    X_nsynth, y_nsynth = prepare_nsynth_from_tfrecord(nsynth_tfrecord, max_samples=3000)
    
    if len(X_nsynth) > 0:
        print(f'\nNSynth dataset: {X_nsynth.shape[0]} samples, {X_nsynth.shape[1]} features')
        print(f'Family indices: {np.unique(y_nsynth)}')
        print(f'Distribution: {pd.Series(y_nsynth).value_counts().sort_index().to_dict()}')
    else:
        print('No samples extracted from NSynth')
        X_nsynth, y_nsynth = None, None
else:
    print('Skipping NSynth dataset (TFRecord not found)')
    X_nsynth, y_nsynth = None, None

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# CELL 13: Train NSynth Family Classifier
# ══════════════════════════════════════════════════════════════════════

if X_nsynth is not None and len(X_nsynth) > 100:
    # y_nsynth already contains family indices (0-10)
    n_nsynth_classes = len(np.unique(y_nsynth))
    print(f'NSynth classes: {n_nsynth_classes} instrument families')
    print(f'Family indices: {sorted(np.unique(y_nsynth))}')
    
    # Filter classes with enough samples
    class_counts = pd.Series(y_nsynth).value_counts()
    valid_classes = class_counts[class_counts >= 20].index.tolist()
    mask = np.isin(y_nsynth, valid_classes)
    X_nsynth_f = X_nsynth[mask]
    y_nsynth_f = y_nsynth[mask]
    
    # Re-encode to consecutive labels
    nsynth_le = LabelEncoder()
    y_nsynth_enc = nsynth_le.fit_transform(y_nsynth_f)
    n_nsynth_classes = len(nsynth_le.classes_)
    print(f'After filtering: {n_nsynth_classes} classes, {len(X_nsynth_f)} samples')
    
    # Split
    X_train, X_test, y_train, y_test = train_test_split(
        X_nsynth_f, y_nsynth_enc, test_size=0.2, random_state=42, stratify=y_nsynth_enc)
    
    # Scale
    nsynth_scaler = StandardScaler()
    X_train_s = nsynth_scaler.fit_transform(X_train)
    X_test_s = nsynth_scaler.transform(X_test)
    
    # Build model
    nsynth_model = keras.Sequential([
        keras.Input(shape=(X_train_s.shape[1],)),
        layers.Dense(256, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.4),
        layers.Dense(128, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.3),
        layers.Dense(64, activation='relu'),
        layers.Dense(n_nsynth_classes, activation='softmax')
    ], name='nsynth_classifier')
    
    nsynth_model.compile(
        optimizer='adam',
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    
    # Train
    nsynth_history = nsynth_model.fit(
        X_train_s, y_train,
        validation_split=0.15,
        epochs=60,
        batch_size=64,
        callbacks=[
            callbacks.EarlyStopping(patience=12, restore_best_weights=True),
            callbacks.ReduceLROnPlateau(patience=5, factor=0.5)
        ],
        verbose=1
    )
    
    # Evaluate
    test_loss, test_acc = nsynth_model.evaluate(X_test_s, y_test, verbose=0)
    print(f'\nNSynth family classifier test accuracy: {test_acc:.4f}')
    
    # Save with family name mapping
    family_mapping = {i: NSYNTH_FAMILIES[c] for i, c in enumerate(nsynth_le.classes_)}
    
    nsynth_model.save('nsynth_classifier.h5')
    joblib.dump(nsynth_scaler, 'nsynth_scaler.pkl')
    joblib.dump(nsynth_le, 'nsynth_labels.pkl')
    joblib.dump(family_mapping, 'nsynth_family_mapping.pkl')
    print('Saved: nsynth_classifier.h5, nsynth_scaler.pkl, nsynth_labels.pkl, nsynth_family_mapping.pkl')
else:
    print('Skipping NSynth model training (insufficient data)')

---

# 4. Onset/Beat Detection Enhancement

**Dataset**: `yuxuanxue/freesound-audio-tagging-2019`

Freesound provides diverse audio with onset annotations useful for enhancing beat detection.

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# CELL 14: Load Freesound Dataset
# ══════════════════════════════════════════════════════════════════════

FREESOUND_PATH = PATHS['freesound']

if os.path.exists(FREESOUND_PATH):
    print('Freesound Audio Tagging 2019 dataset contents:')
    for item in os.listdir(FREESOUND_PATH):
        item_path = os.path.join(FREESOUND_PATH, item)
        if os.path.isfile(item_path):
            size_mb = os.path.getsize(item_path) / 1024 / 1024
            print(f'  {item} ({size_mb:.1f} MB)')
        else:
            count = len(os.listdir(item_path))
            print(f'  {item}/ ({count} items)')
    
    # Load CSV if available
    csv_files = glob.glob(os.path.join(FREESOUND_PATH, '*.csv'))
    if csv_files:
        fs_df = pd.read_csv(csv_files[0])
        print(f'\nCSV shape: {fs_df.shape}')
        print(fs_df.head())
else:
    print(f'⚠ Dataset not found at {FREESOUND_PATH}')
    print('  Add: yuxuanxue/freesound-audio-tagging-2019')

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# CELL 15: Build Onset Feature Dataset
# ══════════════════════════════════════════════════════════════════════

import zipfile

def extract_onset_features(y, sr=SAMPLE_RATE):
    """Extract features useful for onset detection."""
    features = []
    
    # Onset strength envelope
    onset_env = librosa.onset.onset_strength(y=y, sr=sr, hop_length=HOP_LENGTH)
    features.extend([onset_env.mean(), onset_env.std(), onset_env.max()])
    
    # Spectral flux
    S = np.abs(librosa.stft(y, hop_length=HOP_LENGTH))
    flux = np.sqrt(np.mean(np.diff(S, axis=1)**2, axis=0))
    features.extend([flux.mean(), flux.std()])
    
    # RMS dynamics
    rms = librosa.feature.rms(y=y, hop_length=HOP_LENGTH)[0]
    rms_diff = np.diff(rms)
    features.extend([rms.mean(), rms.std(), np.abs(rms_diff).mean()])
    
    # Spectral features
    cent = librosa.feature.spectral_centroid(y=y, sr=sr, hop_length=HOP_LENGTH)[0]
    features.extend([cent.mean(), cent.std()])
    
    # Zero crossings
    zcr = librosa.feature.zero_crossing_rate(y, hop_length=HOP_LENGTH)[0]
    features.extend([zcr.mean(), zcr.std()])
    
    # MFCC deltas (transient detection)
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13, hop_length=HOP_LENGTH)
    mfcc_delta = librosa.feature.delta(mfcc)
    for i in range(13):
        features.extend([mfcc_delta[i].mean(), np.abs(mfcc_delta[i]).mean()])
    
    return np.array(features, dtype=np.float32)


def prepare_onset_dataset_from_zip(base_path, csv_path, max_samples=2000):
    """Prepare onset detection training data from ZIP archive."""
    X, y = [], []
    
    # Load CSV to get filenames
    df = pd.read_csv(csv_path)
    filenames = df['fname'].tolist()
    np.random.shuffle(filenames)
    filenames = filenames[:max_samples]
    
    # Find the curated ZIP (smaller, cleaner data)
    zip_path = os.path.join(base_path, 'train_curated.zip')
    if not os.path.exists(zip_path):
        zip_path = os.path.join(base_path, 'train_noisy.zip')
    
    if not os.path.exists(zip_path):
        print(f'No ZIP file found at {base_path}')
        return np.array([]), np.array([])
    
    print(f'Reading from: {os.path.basename(zip_path)}')
    print(f'Processing up to {len(filenames)} files...')
    
    processed = 0
    with zipfile.ZipFile(zip_path, 'r') as zf:
        # Get list of files in ZIP
        zip_files = {os.path.basename(f): f for f in zf.namelist() if f.endswith('.wav')}
        
        for fname in tqdm(filenames):
            if processed >= max_samples:
                break
            
            if fname not in zip_files:
                continue
            
            try:
                # Read audio from ZIP
                with zf.open(zip_files[fname]) as audio_file:
                    import io
                    audio_bytes = io.BytesIO(audio_file.read())
                    audio, sr = librosa.load(audio_bytes, sr=SAMPLE_RATE, duration=5.0, mono=True)
                
                if audio is None or len(audio) < SAMPLE_RATE:
                    continue
                
                # Get librosa onset times
                onsets = librosa.onset.onset_detect(
                    y=audio, sr=SAMPLE_RATE, hop_length=HOP_LENGTH, units='time')
                
                # Extract features
                features = extract_onset_features(audio)
                
                if not np.any(np.isnan(features)) and not np.any(np.isinf(features)):
                    X.append(features)
                    # Binary: has strong onsets (percussive) vs weak (sustained)
                    has_strong_onsets = len(onsets) > 5
                    y.append(1 if has_strong_onsets else 0)
                    processed += 1
            except Exception as e:
                pass
    
    return np.array(X), np.array(y)


if os.path.exists(FREESOUND_PATH):
    # Use train_curated.csv
    csv_path = os.path.join(FREESOUND_PATH, 'train_curated.csv')
    if not os.path.exists(csv_path):
        csv_path = os.path.join(FREESOUND_PATH, 'train_noisy.csv')
    
    X_onset, y_onset = prepare_onset_dataset_from_zip(FREESOUND_PATH, csv_path, max_samples=2000)
    
    if len(X_onset) > 0:
        print(f'\nOnset dataset: {X_onset.shape[0]} samples, {X_onset.shape[1]} features')
        print(f'Class distribution: {np.bincount(y_onset)}')
    else:
        print('No samples extracted')
        X_onset, y_onset = None, None
else:
    print('Skipping Freesound (not available)')
    X_onset, y_onset = None, None

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# CELL 16: Train Onset Detector
# ══════════════════════════════════════════════════════════════════════

if X_onset is not None and len(X_onset) > 100:
    # Split
    X_train, X_test, y_train, y_test = train_test_split(
        X_onset, y_onset, test_size=0.2, random_state=42, stratify=y_onset)
    
    # Scale
    onset_scaler = StandardScaler()
    X_train_s = onset_scaler.fit_transform(X_train)
    X_test_s = onset_scaler.transform(X_test)
    
    # Build model
    onset_model = keras.Sequential([
        keras.Input(shape=(X_train_s.shape[1],)),
        layers.Dense(64, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.3),
        layers.Dense(32, activation='relu'),
        layers.Dense(1, activation='sigmoid')
    ], name='onset_detector')
    
    onset_model.compile(
        optimizer='adam',
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    
    # Train
    onset_history = onset_model.fit(
        X_train_s, y_train,
        validation_split=0.15,
        epochs=40,
        batch_size=32,
        callbacks=[
            callbacks.EarlyStopping(patience=8, restore_best_weights=True)
        ],
        verbose=1
    )
    
    # Evaluate
    test_loss, test_acc = onset_model.evaluate(X_test_s, y_test, verbose=0)
    print(f'\nOnset detector test accuracy: {test_acc:.4f}')
    
    # Save
    onset_model.save('onset_detector.h5')
    joblib.dump(onset_scaler, 'onset_scaler.pkl')
    print('Saved: onset_detector.h5, onset_scaler.pkl')
else:
    print('Skipping onset model training (insufficient data)')

---

# 5. Multi-Label Tag Predictor (Mixing Intelligence)

**Dataset**: `shrirangmahajan/magnatagatune`

MagnaTagATune provides music clips with multiple tags (mood, instruments, tempo)—useful for intelligent mixing decisions.

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# CELL 17: Load MagnaTagATune Dataset
# ══════════════════════════════════════════════════════════════════════

MAGNA_PATH = PATHS['magnatagatune']

if os.path.exists(MAGNA_PATH):
    print('MagnaTagATune dataset contents:')
    for item in os.listdir(MAGNA_PATH):
        print(f'  {item}')
    
    # Load annotations
    annotation_files = glob.glob(os.path.join(MAGNA_PATH, '**', '*annotation*'), recursive=True)
    annotation_files += glob.glob(os.path.join(MAGNA_PATH, '**', '*tags*'), recursive=True)
    print(f'\nAnnotation files found: {annotation_files}')
    
    if annotation_files:
        # Try to load the first annotation file
        for af in annotation_files:
            try:
                if af.endswith('.csv'):
                    magna_df = pd.read_csv(af, sep='\t' if 'tsv' in af else ',')
                else:
                    magna_df = pd.read_csv(af, sep='\t')
                print(f'\nLoaded: {af}')
                print(f'Shape: {magna_df.shape}')
                print(magna_df.head())
                break
            except:
                continue
else:
    print(f'⚠ Dataset not found at {MAGNA_PATH}')
    print('  Add: shrirangmahajan/magnatagatune')

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# CELL 18: Prepare MagnaTagATune Features
# ══════════════════════════════════════════════════════════════════════

def prepare_magnatagatune_dataset(base_path, max_samples=1500):
    """Prepare multi-label tag prediction dataset."""
    X, y_tags = [], []
    
    # Find annotation file
    annotation_files = glob.glob(os.path.join(base_path, '**', '*annotation*'), recursive=True)
    annotation_files += glob.glob(os.path.join(base_path, '**', '*tags*'), recursive=True)
    
    if not annotation_files:
        print('No annotation files found')
        return None, None, None
    
    print(f'Found annotation files: {[os.path.basename(f) for f in annotation_files]}')
    
    # Load annotations - try different separators
    df = None
    for af in annotation_files:
        for sep in ['\t', ',', ';']:
            try:
                df = pd.read_csv(af, sep=sep)
                if len(df.columns) > 10:
                    print(f'Loaded: {os.path.basename(af)} with sep={repr(sep)}')
                    print(f'Shape: {df.shape}')
                    break
                else:
                    df = None
            except:
                df = None
        if df is not None and len(df.columns) > 10:
            break
    
    if df is None or len(df.columns) < 10:
        print('Could not load annotations properly')
        return None, None, None
    
    print(f'Columns (first 10): {df.columns.tolist()[:10]}')
    print(f'Columns (last 5): {df.columns.tolist()[-5:]}')
    
    # Find the mp3_path column (prefer 'mp3_path' over 'clip_id')
    path_col = None
    for col in df.columns:
        if 'mp3_path' in col.lower() or 'path' in col.lower():
            path_col = col
            break
    
    if path_col is None:
        # Fallback to last column or clip_id
        path_col = df.columns[-1]
    
    print(f'Path column: {path_col}')
    print(f'Sample paths: {df[path_col].head(3).tolist()}')
    
    # Find audio files
    audio_files = glob.glob(os.path.join(base_path, '**', '*.mp3'), recursive=True)
    audio_files += glob.glob(os.path.join(base_path, '**', '*.wav'), recursive=True)
    
    print(f'Found {len(audio_files)} audio files')
    if audio_files:
        print(f'Sample audio paths: {[os.path.basename(f) for f in audio_files[:3]]}')
    
    # Create flexible filename to path mapping
    file_map = {}
    for f in audio_files:
        basename = os.path.basename(f)
        # Map by basename
        file_map[basename] = f
        # Map by basename without extension
        file_map[os.path.splitext(basename)[0]] = f
        # Map by relative path from base
        rel_path = os.path.relpath(f, base_path).replace('\\', '/')
        file_map[rel_path] = f
        # Map by relative path without extension
        file_map[os.path.splitext(rel_path)[0]] = f
    
    # Determine tag columns (binary 0/1 columns, excluding path and id columns)
    exclude_cols = [path_col, 'clip_id', 'mp3_path']
    tag_cols = []
    for c in df.columns:
        if c in exclude_cols:
            continue
        try:
            unique_vals = df[c].dropna().unique()
            if len(unique_vals) <= 2 and all(v in [0, 1, 0.0, 1.0] for v in unique_vals):
                tag_cols.append(c)
        except:
            pass
    
    print(f'Tag columns found: {len(tag_cols)}')
    if tag_cols:
        print(f'Sample tags: {tag_cols[:10]}')
    
    if len(tag_cols) == 0:
        print('No valid tag columns found')
        return None, None, None
    
    # Process samples
    samples_processed = 0
    skipped_not_found = 0
    
    # Shuffle to get diverse samples
    indices = df.index.tolist()
    np.random.shuffle(indices)
    
    for idx in tqdm(indices[:max_samples * 3], desc='Processing audio'):
        if samples_processed >= max_samples:
            break
        
        row = df.loc[idx]
        
        # Get the mp3 path from the annotation
        mp3_path = str(row[path_col]).strip().strip('"')
        
        # Try different path formats to find the file
        audio_path = None
        candidates = [
            mp3_path,
            os.path.basename(mp3_path),
            os.path.splitext(os.path.basename(mp3_path))[0],
            mp3_path.replace('.mp3', ''),
            # Try with .mp3 extension added
            mp3_path + '.mp3' if not mp3_path.endswith('.mp3') else mp3_path,
        ]
        
        for key in candidates:
            if key in file_map:
                audio_path = file_map[key]
                break
        
        if audio_path is None:
            skipped_not_found += 1
            if skipped_not_found <= 3:
                print(f'  Not found: {mp3_path}')
            continue
        
        audio = load_audio_safe(audio_path, duration=10.0)
        if audio is None or len(audio) < SAMPLE_RATE * 2:
            continue
        
        try:
            features = extract_audio_features(audio)
            if not np.any(np.isnan(features)) and not np.any(np.isinf(features)):
                X.append(features)
                tags = [int(row[c]) if pd.notna(row[c]) else 0 for c in tag_cols]
                y_tags.append(tags)
                samples_processed += 1
        except Exception as e:
            pass
    
    print(f'\nProcessed: {samples_processed}, Skipped (not found): {skipped_not_found}')
    
    if len(X) == 0:
        return None, None, None
    
    return np.array(X), np.array(y_tags), tag_cols


if os.path.exists(MAGNA_PATH):
    X_magna, y_magna, magna_tags = prepare_magnatagatune_dataset(MAGNA_PATH)
    if X_magna is not None and len(X_magna) > 0:
        print(f'\nMagnaTagATune dataset: {X_magna.shape[0]} samples, {X_magna.shape[1]} features')
        print(f'Tags: {len(magna_tags)} labels')
    else:
        print('No samples extracted from MagnaTagATune')
        X_magna, y_magna, magna_tags = None, None, None
else:
    print('Skipping MagnaTagATune (not available)')
    X_magna, y_magna, magna_tags = None, None, None

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# CELL 19: Train Multi-Label Tag Predictor
# ══════════════════════════════════════════════════════════════════════

if X_magna is not None and len(X_magna) > 100:
    # Remove tags with too few positives
    tag_sums = y_magna.sum(axis=0)
    valid_tags = tag_sums >= 20
    y_magna_f = y_magna[:, valid_tags]
    magna_tags_f = [t for t, v in zip(magna_tags, valid_tags) if v]
    
    print(f'Filtered to {y_magna_f.shape[1]} valid tags')
    
    # Split
    X_train, X_test, y_train, y_test = train_test_split(
        X_magna, y_magna_f, test_size=0.2, random_state=42)
    
    # Scale
    tag_scaler = StandardScaler()
    X_train_s = tag_scaler.fit_transform(X_train)
    X_test_s = tag_scaler.transform(X_test)
    
    # Build multi-label model
    n_tags = y_magna_f.shape[1]
    
    tag_model = keras.Sequential([
        keras.Input(shape=(X_train_s.shape[1],)),
        layers.Dense(256, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.4),
        layers.Dense(128, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.3),
        layers.Dense(64, activation='relu'),
        layers.Dense(n_tags, activation='sigmoid')  # Multi-label: sigmoid for each tag
    ], name='tag_predictor')
    
    tag_model.compile(
        optimizer='adam',
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    
    # Train
    tag_history = tag_model.fit(
        X_train_s, y_train,
        validation_split=0.15,
        epochs=50,
        batch_size=32,
        callbacks=[
            callbacks.EarlyStopping(patience=10, restore_best_weights=True)
        ],
        verbose=1
    )
    
    # Evaluate
    y_pred = (tag_model.predict(X_test_s, verbose=0) > 0.5).astype(int)
    f1 = f1_score(y_test, y_pred, average='micro')
    print(f'\nTag predictor micro F1: {f1:.4f}')
    
    # Save
    tag_model.save('tag_predictor.h5')
    joblib.dump(tag_scaler, 'tag_scaler.pkl')
    joblib.dump(magna_tags_f, 'tag_labels.pkl')
    print('Saved: tag_predictor.h5, tag_scaler.pkl, tag_labels.pkl')
else:
    print('Skipping tag model training (insufficient data)')

---

# 6. Spotify Million Song Dataset for Mixing Intelligence

**Dataset**: `notshrirang/spotify-million-song-dataset`

Extracts song metadata (artist, lyrics, etc.) for enhanced mixing recommendations.

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# CELL 20: Load Spotify Million Song Dataset
# ══════════════════════════════════════════════════════════════════════

SPOTIFY_PATH = PATHS['spotify_msd']

if os.path.exists(SPOTIFY_PATH):
    print('Spotify Million Song Dataset contents:')
    for item in os.listdir(SPOTIFY_PATH)[:20]:
        item_path = os.path.join(SPOTIFY_PATH, item)
        if os.path.isfile(item_path):
            size_mb = os.path.getsize(item_path) / 1024 / 1024
            print(f'  {item} ({size_mb:.1f} MB)')
        else:
            count = len(os.listdir(item_path)) if os.path.isdir(item_path) else 0
            print(f'  {item}/ ({count} items)')
    
    # Look for CSV files
    csv_files = glob.glob(os.path.join(SPOTIFY_PATH, '**', '*.csv'), recursive=True)
    print(f'\nFound {len(csv_files)} CSV files')
    
    if csv_files:
        sample_df = pd.read_csv(csv_files[0], nrows=5)
        print(f'\nSample columns: {sample_df.columns.tolist()}')
        print(sample_df.head())
else:
    print(f'⚠ Dataset not found at {SPOTIFY_PATH}')
    print('  Add: notshrirang/spotify-million-song-dataset')

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# CELL 21: Process Spotify Metadata for Mixing Intelligence
# ══════════════════════════════════════════════════════════════════════

SPOTIFY_PATH = PATHS['spotify_msd']

def process_spotify_metadata(spotify_path):
    """Process Spotify Million Song Dataset for mixing features."""
    csv_files = glob.glob(os.path.join(spotify_path, '**', '*.csv'), recursive=True)
    
    if not csv_files:
        print('No CSV files found')
        return None, None
    
    # Load all CSVs
    dfs = []
    for cf in csv_files:
        try:
            df = pd.read_csv(cf)
            dfs.append(df)
            print(f'Loaded: {os.path.basename(cf)} ({len(df)} rows)')
        except Exception as e:
            print(f'Error loading {cf}: {e}')
    
    if not dfs:
        return None, None
    
    spotify_df = pd.concat(dfs, ignore_index=True)
    print(f'\nTotal songs: {len(spotify_df)}')
    print(f'Columns: {spotify_df.columns.tolist()}')
    
    # Extract useful metadata for mixing
    metadata = {
        'total_songs': len(spotify_df),
        'columns': spotify_df.columns.tolist(),
    }
    
    # If there's artist info, get unique artists
    artist_cols = [c for c in spotify_df.columns if 'artist' in c.lower()]
    if artist_cols:
        metadata['unique_artists'] = spotify_df[artist_cols[0]].nunique()
        metadata['top_artists'] = spotify_df[artist_cols[0]].value_counts().head(20).to_dict()
    
    # If there's song/track info
    song_cols = [c for c in spotify_df.columns if any(x in c.lower() for x in ['song', 'track', 'title'])]
    if song_cols:
        metadata['sample_songs'] = spotify_df[song_cols[0]].head(100).tolist()
    
    # If there's lyrics, analyze text features
    lyrics_cols = [c for c in spotify_df.columns if 'lyric' in c.lower() or 'text' in c.lower()]
    if lyrics_cols:
        lyrics_df = spotify_df[lyrics_cols[0]].dropna()
        metadata['songs_with_lyrics'] = len(lyrics_df)
        metadata['avg_lyric_length'] = lyrics_df.str.len().mean()
    
    return metadata, spotify_df


if os.path.exists(SPOTIFY_PATH):
    spotify_meta, spotify_df = process_spotify_metadata(SPOTIFY_PATH)
    if spotify_meta:
        print(f'\n--- Spotify Dataset Summary ---')
        print(f"Total songs: {spotify_meta.get('total_songs', 'N/A')}")
        print(f"Unique artists: {spotify_meta.get('unique_artists', 'N/A')}")
        print(f"Songs with lyrics: {spotify_meta.get('songs_with_lyrics', 'N/A')}")
        
        # Save metadata
        joblib.dump(spotify_meta, 'song_metadata.pkl')
        print('\nSaved: song_metadata.pkl')
else:
    print(f'⚠ Spotify MSD not found at {SPOTIFY_PATH}')
    print('  Add: notshrirang/spotify-million-song-dataset')

---

# 7. Lakh MIDI Dataset for Generation

**Dataset**: `federicodellellis/lakh-midi-dataset-clean`

Analyzes MIDI patterns for improved beat generation.

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# CELL 22: Load Lakh MIDI Dataset
# ══════════════════════════════════════════════════════════════════════

LAKH_PATH = PATHS['lakh_midi']

if os.path.exists(LAKH_PATH):
    print('Lakh MIDI Dataset Clean contents:')
    for item in os.listdir(LAKH_PATH)[:15]:
        print(f'  {item}')
    
    midi_files = glob.glob(os.path.join(LAKH_PATH, '**', '*.mid'), recursive=True)
    midi_files += glob.glob(os.path.join(LAKH_PATH, '**', '*.midi'), recursive=True)
    print(f'\nFound {len(midi_files)} MIDI files')
else:
    print(f'⚠ Dataset not found at {LAKH_PATH}')
    print('  Add: federicodellellis/lakh-midi-dataset-clean')

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# CELL 23: Analyze MIDI Patterns
# ══════════════════════════════════════════════════════════════════════

try:
    import pretty_midi
    PRETTY_MIDI_AVAILABLE = True
except ImportError:
    PRETTY_MIDI_AVAILABLE = False
    print('pretty_midi not available, skipping MIDI analysis')


def analyze_midi_patterns(lakh_path, max_files=500):
    """Extract rhythm patterns from MIDI files for beat generation."""
    if not PRETTY_MIDI_AVAILABLE:
        return None
    
    patterns = []
    tempos = []
    time_sigs = []
    
    midi_files = glob.glob(os.path.join(lakh_path, '**', '*.mid'), recursive=True)
    midi_files += glob.glob(os.path.join(lakh_path, '**', '*.midi'), recursive=True)
    np.random.shuffle(midi_files)
    midi_files = midi_files[:max_files]
    
    print(f'Analyzing {len(midi_files)} MIDI files...')
    
    for mf in tqdm(midi_files):
        try:
            midi = pretty_midi.PrettyMIDI(mf)
            
            # Extract tempo
            tempo_changes = midi.get_tempo_changes()
            if len(tempo_changes[1]) > 0:
                tempos.append(tempo_changes[1][0])  # First tempo
            
            # Extract time signature
            for ts in midi.time_signature_changes:
                time_sigs.append((ts.numerator, ts.denominator))
            
            # Extract drum patterns (channel 9 / program 0-127 drums)
            for instrument in midi.instruments:
                if instrument.is_drum:
                    # Get note onsets
                    onsets = [note.start for note in instrument.notes]
                    pitches = [note.pitch for note in instrument.notes]
                    velocities = [note.velocity for note in instrument.notes]
                    
                    if len(onsets) >= 8:
                        patterns.append({
                            'onsets': onsets[:64],
                            'pitches': pitches[:64],
                            'velocities': velocities[:64]
                        })
        except:
            pass
    
    return {
        'patterns': patterns,
        'tempos': tempos,
        'time_signatures': time_sigs
    }


if os.path.exists(LAKH_PATH) and PRETTY_MIDI_AVAILABLE:
    midi_analysis = analyze_midi_patterns(LAKH_PATH)
    if midi_analysis:
        print(f'\nExtracted {len(midi_analysis["patterns"])} drum patterns')
        print(f'Tempo range: {np.min(midi_analysis["tempos"]):.0f} - {np.max(midi_analysis["tempos"]):.0f} BPM')
        print(f'Most common time sigs: {pd.Series(midi_analysis["time_signatures"]).value_counts().head()}')
        
        # Save patterns
        joblib.dump(midi_analysis, 'midi_patterns.pkl')
        print('Saved: midi_patterns.pkl')
else:
    print('Skipping Lakh MIDI (not available)')

---

# Summary & Output Files

This notebook trains/generates the following models and artifacts:

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# CELL 24: Summary of Generated Files
# ══════════════════════════════════════════════════════════════════════

output_files = [
    ('drum_classifier.h5', 'Drum type classifier (kick/snare/hihat/etc)'),
    ('drum_scaler.pkl', 'Scaler for drum features'),
    ('drum_labels.pkl', 'Label encoder for drum classes'),
    ('nsynth_classifier.h5', 'NSynth instrument family classifier'),
    ('nsynth_scaler.pkl', 'Scaler for NSynth features'),
    ('nsynth_labels.pkl', 'Label encoder for NSynth families'),
    ('onset_detector.h5', 'Onset/transient detector'),
    ('onset_scaler.pkl', 'Scaler for onset features'),
    ('tag_predictor.h5', 'Multi-label music tag predictor'),
    ('tag_scaler.pkl', 'Scaler for tag features'),
    ('tag_labels.pkl', 'List of tag labels'),
    ('midi_patterns.pkl', 'Extracted MIDI rhythm patterns'),
    ('song_metadata.pkl', 'Spotify song metadata for mixing'),
]

print('=' * 60)
print('OUTPUT FILES')
print('=' * 60)
print()

for filename, description in output_files:
    exists = os.path.exists(filename)
    status = '✓' if exists else '✗'
    size = f'{os.path.getsize(filename) / 1024:.1f} KB' if exists else 'N/A'
    print(f'{status} {filename:30} {size:>10}  │ {description}')

print()
print('=' * 60)
print('DEPLOYMENT')
print('=' * 60)
print('''
1. Download all generated .h5 and .pkl files from Kaggle output
2. Copy to: backend/app/storage/models/
3. Update backend services to load new models:
   - drum_classifier.py (new)
   - nsynth_classifier.py (new)
   - onset_detector.py (new)
   - tag_predictor.py (new)
   - song_matcher.py (new) - uses song_metadata.pkl
4. Restart the AutoMixAI server
''')

---

## Dataset Reference (Quick Copy)

Add these datasets to your Kaggle notebook:

| Purpose | Kaggle Dataset Path |
|---------|---------------------|
| Beat/Onset Detection | `yuxuanxue/freesound-audio-tagging-2019` |
| Instrument Classification | `anubhavchhabra/nsynth-music-dataset` |
| Drum Classification | `sparshgupta/drum-kit-sound-samples` |
| MIDI Patterns (Generation) | `federicodellellis/lakh-midi-dataset-clean` |
| Multi-Label Tags (Mixing) | `shrirangmahajan/magnatagatune` |
| Song Metadata (Mixing) | `notshrirang/spotify-million-song-dataset` |

**Already used (skip these):** Ballroom, FMA, GTZAN